# Research Paper Intelligence System

An end-to-end NLP pipeline that lets you **search, summarize, extract keywords from, and compare**
research papers using natural language queries.

**Pipeline overview**
1. Load and clean a corpus of ML research papers (title + abstract)
2. Generate semantic embeddings with Sentence-Transformers
3. Build a FAISS vector index for fast semantic search
4. Summarize retrieved papers with a BART summarization model
5. Extract keyphrases with KeyBERT
6. Wrap everything as tools for an LLM agent (LangChain + Groq) so it can be queried conversationally

**Tech stack:** `sentence-transformers`, `faiss`, `transformers`, `keybert`, `langchain`, `langchain-groq`


## 1. Setup

In [3]:
%pip install -q datasets sentence-transformers faiss-cpu keybert \
    transformers==4.46.3 huggingface_hub==0.26.2 tokenizers==0.20.3 sentence-transformers==3.3.1 \
    langchain langchain-community langchain-core langchain-huggingface langchain-groq

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip
error: resolution-too-deep

× Dependency resolution exceeded maximum depth
╰─> Pip cannot resolve the current dependencies as the dependency graph is too complex for pip to solve efficiently.

hint: Try adding lower bounds to constrain your dependencies, for example: 'package>=2.0.0' instead of just 'package'.

Link: https://pip.pypa.io/en/stable/topics/dependency-resolution/#handling-resolution-too-deep-errors


In [6]:
%pip install -q datasets

import os
import numpy as np
import pandas as pd

from datasets import load_dataset
%pip install -q datasets

import os
import numpy as np
import pandas as pd

from datasets import load_dataset
%pip install -q datasets sentence-transformers faiss-cpu transformers keybert

import os
import numpy as np
import pandas as pd
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import faiss
from transformers import pipeline
from keybert import KeyBERT

pd.set_option("display.max_colwidth", 120)
from sklearn.metrics.pairwise import cosine_similarity
import faiss
from transformers import pipeline
from keybert import KeyBERT

pd.set_option("display.max_colwidth", 120)
from sklearn.metrics.pairwise import cosine_similarity
import faiss
from transformers import pipeline
from keybert import KeyBERT

pd.set_option("display.max_colwidth", 120)

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## 2. Load and Prepare the Dataset

We use the `CShorten/ML-ArXiv-Papers` dataset from Hugging Face, which contains titles and
abstracts of machine learning papers from arXiv. For a manageable demo footprint, we work with
a subset of the data; the same pipeline scales to the full dataset given enough compute/storage.

In [7]:
dataset = load_dataset("CShorten/ML-ArXiv-Papers", split="train")

df = pd.DataFrame(dataset)[["title", "abstract"]]

SAMPLE_SIZE = 15000
df = df.head(SAMPLE_SIZE).reset_index(drop=True)

print(f"Loaded {len(df)} papers")
df.head()

Generating train split: 100%|██████████| 117592/117592 [00:02<00:00, 51972.76 examples/s]


Loaded 15000 papers


,title,abstract
0,Learning from compressed observations,The problem of statistical learning is to construct a predictor of a random\nvariable $Y$ as a function of a relat...
1,Sensor Networks with Random Links: Topology Design for Distributed\n Consensus,"In a sensor network, in practice, the communication among sensors is subject\nto:(1) errors or failures at random ..."
2,The on-line shortest path problem under partial monitoring,The on-line shortest path problem is considered under various models of\npartial monitoring. Given a weighted dire...
3,A neural network approach to ordinal regression,"Ordinal regression is an important type of learning, which has properties of\nboth classification and regression. ..."
4,Parametric Learning and Monte Carlo Optimization,This paper uncovers and explores the close relationship between Monte Carlo\nOptimization of a parametrized integr...


In [8]:
# Basic cleaning
df = df.dropna(subset=["title", "abstract"]).reset_index(drop=True)
df = df.drop_duplicates(subset=["title"]).reset_index(drop=True)

df["paper_text"] = (df["title"] + " " + df["abstract"]).str.replace("\n", " ", regex=False).str.strip()

print(f"{len(df)} papers remaining after cleaning")
df[["title", "paper_text"]].head(3)

14964 papers remaining after cleaning


,title,paper_text
0,Learning from compressed observations,Learning from compressed observations The problem of statistical learning is to construct a predictor of a random ...
1,Sensor Networks with Random Links: Topology Design for Distributed\n Consensus,"Sensor Networks with Random Links: Topology Design for Distributed Consensus In a sensor network, in practice, t..."
2,The on-line shortest path problem under partial monitoring,The on-line shortest path problem under partial monitoring The on-line shortest path problem is considered under v...


## 3. Semantic Embeddings

We use `all-MiniLM-L6-v2`, a lightweight but strong sentence-embedding model, to turn each
paper's combined title + abstract into a 384-dimensional vector. Papers with similar meaning
end up close together in this vector space, which is what makes semantic search possible
(as opposed to plain keyword matching).

In [9]:
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

# Quick sanity check: similarity between the first paper and a few others
sample_embeddings = embed_model.encode(df["paper_text"].head(5).tolist())

for i in range(1, 5):
    sim = cosine_similarity(sample_embeddings[0].reshape(1, -1), sample_embeddings[i].reshape(1, -1))
    print(f"Paper 1 vs Paper {i + 1}: {sim[0][0]:.4f}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2810.99it/s]


Paper 1 vs Paper 2: 0.3663
Paper 1 vs Paper 3: 0.3352
Paper 1 vs Paper 4: 0.1551
Paper 1 vs Paper 5: 0.3742


In [10]:
EMBEDDINGS_PATH = "paper_embeddings.npy"

if os.path.exists(EMBEDDINGS_PATH):
    print("Loading cached embeddings...")
    embeddings = np.load(EMBEDDINGS_PATH)
else:
    print("Generating embeddings for the full corpus (this may take a few minutes)...")
    embeddings = embed_model.encode(
        df["paper_text"].tolist(),
        batch_size=32,
        show_progress_bar=True,
    )
    np.save(EMBEDDINGS_PATH, embeddings)
    print("Embeddings saved.")

print(f"Embeddings shape: {embeddings.shape}  |  Size: {embeddings.nbytes / (1024 * 1024):.1f} MB")

Generating embeddings for the full corpus (this may take a few minutes)...


Batches: 100%|██████████| 468/468 [10:09<00:00,  1.30s/it]


Embeddings saved.
Embeddings shape: (14964, 384)  |  Size: 21.9 MB


## 4. Vector Index (FAISS)

The embeddings are indexed with FAISS using an inner-product index over L2-normalized vectors,
which is equivalent to cosine similarity search. This gives fast approximate/exact nearest-neighbor
lookup even as the corpus grows.

In [11]:
INDEX_PATH = "paper_faiss.index"
EMBED_DIM = embeddings.shape[1]

if os.path.exists(INDEX_PATH):
    print("Loading cached FAISS index...")
    index = faiss.read_index(INDEX_PATH)
else:
    print("Building FAISS index...")
    faiss.normalize_L2(embeddings)
    index = faiss.IndexFlatIP(EMBED_DIM)
    index.add(embeddings)
    faiss.write_index(index, INDEX_PATH)
    print("Index built and saved.")

print(f"Vectors indexed: {index.ntotal}")

Building FAISS index...
Index built and saved.
Vectors indexed: 14964


In [12]:
def search_papers(query: str, k: int = 5):
    """Return the top-k papers most semantically similar to the query."""
    if not query or not query.strip():
        raise ValueError("Query must be a non-empty string")

    query_embedding = embed_model.encode([query])
    faiss.normalize_L2(query_embedding)

    scores, indices = index.search(query_embedding, k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        row = df.iloc[idx]
        results.append({
            "score": round(float(score), 4),
            "title": row["title"],
            "abstract": row["abstract"],
            "index": int(idx),
        })
    return results


def display_results(results):
    for r in results:
        print("=" * 100)
        print(f"Similarity: {r['score']}")
        print(f"Title: {r['title']}")
        print()

In [13]:
display_results(search_papers("deep learning for medical image analysis", k=5))

Similarity: 0.6807
Title: A Perspective on Deep Imaging

Similarity: 0.6709
Title: Convolutional Neural Networks for Medical Image Analysis: Full Training
  or Fine Tuning?

Similarity: 0.6522
Title: Classification of MRI data using Deep Learning and Gaussian
  Process-based Model Selection

Similarity: 0.6281
Title: High-Resolution Breast Cancer Screening with Multi-View Deep
  Convolutional Neural Networks

Similarity: 0.6131
Title: Deep Learning for the Classification of Lung Nodules



## 5. Summarization

Retrieved abstracts are condensed with `facebook/bart-large-cnn`. Summary length is scaled to the
input length so short abstracts aren't padded and long ones aren't cut off awkwardly.

In [10]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")
summarizer_model = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-large-cnn")

def summarize_text(text: str) -> str:
    """Summarize a block of text, scaling target length to the input length."""
    word_count = len(text.split())
    if word_count < 20:
        return text  # too short to meaningfully summarize

    max_len = min(120, max(40, word_count // 2))
    min_len = min(40, max_len - 10)

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=1024,
    )

    summary_ids = summarizer_model.generate(
        **inputs,
        max_new_tokens=max_len,
        min_length=min_len,
        no_repeat_ngram_size=3,
        length_penalty=2.0,
        early_stopping=True,
    )

    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 
Loading weights: 100%|██████████| 511/511 [00:00<00:00, 3968.29it/s]


In [15]:
import faiss

def search_and_summarize(query: str, k: int = 5):
    """Search for papers and return each with a generated summary."""
    results = search_papers(query, k=k)
    for r in results:
        r["summary"] = summarize_text(r["abstract"])
    return results


def display_search_and_summarize(query, k=5):
    for r in search_and_summarize(query, k=k):
        print("=" * 100)
        print(f"Similarity: {r['score']}")
        print(f"Title: {r['title']}")
        print(f"Summary: {r['summary']}")
        print()



if "search_papers" not in globals():

    def search_papers(query: str, k: int = 5):
        """Return the top-k papers most semantically similar to the query."""
        if not query or not query.strip():
            raise ValueError("Query must be a non-empty string")

        if "embed_model" not in globals() or "index" not in globals() or "df" not in globals():
            raise NameError("Run the embedding/index setup cells first so 'embed_model', 'index', and 'df' exist.")

        query_embedding = embed_model.encode([query])
        faiss.normalize_L2(query_embedding)

        scores, indices = index.search(query_embedding, k)
        results = []
        for score, idx in zip(scores[0], indices[0]):
            if idx == -1:
                continue
            row = df.iloc[idx]
            results.append({
                "score": round(float(score), 4),
                "title": row["title"],
                "abstract": row["abstract"],
                "index": int(idx),
            })
        return results

def search_and_summarize(query: str, k: int = 5):
    """Search for papers and return each with a generated summary."""
    results = search_papers(query, k=k)
    for r in results:
        r["summary"] = summarize_text(r["abstract"])
    return results


def display_search_and_summarize(query, k=5):
    for r in search_and_summarize(query, k=k):
        print("=" * 100)
        print(f"Similarity: {r['score']}")
        print(f"Title: {r['title']}")
        print(f"Summary: {r['summary']}")
        print()



if "search_papers" not in globals():

    def search_papers(query: str, k: int = 5):
        """Return the top-k papers most semantically similar to the query."""
        if not query or not query.strip():
            raise ValueError("Query must be a non-empty string")

        if "embed_model" not in globals() or "index" not in globals() or "df" not in globals():
            raise NameError("Run the embedding/index setup cells first so 'embed_model', 'index', and 'df' exist.")

        query_embedding = embed_model.encode([query])
        faiss.normalize_L2(query_embedding)

        scores, indices = index.search(query_embedding, k)
        results = []
        for score, idx in zip(scores[0], indices[0]):
            if idx == -1:
                continue
            row = df.iloc[idx]
            results.append({
                "score": round(float(score), 4),
                "title": row["title"],
                "abstract": row["abstract"],
                "index": int(idx),
            })
        return results

# Only run the demo call if embeddings/index are available
if all(name in globals() for name in ("embed_model", "index", "df")):
    display_search_and_summarize("Deep Learning in Medical Imaging", k=5)
else:
    print("Embeddings/index not available. Run the embedding and FAISS index cells (e.g., cells 8-11) first.")

Embeddings/index not available. Run the embedding and FAISS index cells (e.g., cells 8-11) first.


## 6. Keyword Extraction

`KeyBERT` reuses the same embedding model to pull out the most representative keyphrases from a
paper's abstract, which is useful for quick tagging or topic labeling.

In [19]:

try:
    kw = kw_model
except NameError:
    from keybert import KeyBERT
    kw = KeyBERT(model="all-MiniLM-L6-v2")



if "df" not in globals():
    from datasets import load_dataset
    import pandas as pd

    dataset = load_dataset("CShorten/ML-ArXiv-Papers", split="train")
    df = pd.DataFrame(dataset)[["title", "abstract"]]
    df = df.dropna(subset=["title", "abstract"]).drop_duplicates(subset=["title"]).reset_index(drop=True)
    df = df.head(1000).reset_index(drop=True)


def extract_keywords(text: str, top_n: int = 5):
    """Extract the top-n keyphrases from a piece of text."""
    return kw.extract_keywords(
        text,
        keyphrase_ngram_range=(1, 3),
        stop_words="english",
        top_n=top_n,
    )


sample_abstract = df.iloc[0]["abstract"]
extract_keywords(sample_abstract, top_n=5)

[('information theoretic characterization', 0.5784),
 ('characterization achievable predictor', 0.5384),
 ('function information theoretic', 0.5281),
 ('information theoretic', 0.4972),
 ('distribution allowable predictors', 0.4931)]

## 7. Named Entity Recognition 

A general-purpose NER pipeline can additionally pull out named entities (models, datasets,
institutions) mentioned in a paper, which is useful for building structured metadata.

In [20]:
ner = pipeline("ner", aggregation_strategy="simple")

sample_text = "ResNet50 was trained on ImageNet using PyTorch at Stanford University."
ner(sample_text)

[transformers] No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496.
Using a pipeline without specifying a model name and revision in production is not recommended.
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 2125.47it/s]
[transformers] BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'entity_group': 'ORG',
  'score': np.float32(0.89313793),
  'word': 'ResNet50',
  'start': 0,
  'end': 8},
 {'entity_group': 'ORG',
  'score': np.float32(0.82662475),
  'word': 'ImageNet',
  'start': 24,
  'end': 32},
 {'entity_group': 'ORG',
  'score': np.float32(0.9451176),
  'word': 'PyTorch',
  'start': 39,
  'end': 46},
 {'entity_group': 'ORG',
  'score': np.float32(0.9337441),
  'word': 'Stanford University',
  'start': 50,
  'end': 69}]

## 8. Retrieval Quality Check

Before wiring the pipeline into an agent, it's worth sanity-checking retrieval quality with a
few hand-picked queries and inspecting whether the top results are topically on point.

In [23]:
import faiss
from sentence_transformers import SentenceTransformer

eval_queries = [
    "vision transformers for image classification",
    "reinforcement learning for robotics",
    "graph neural networks",
]

for q in eval_queries:
    print(f"\nQuery: {q}")
    # Rebuild the semantic index if the earlier setup cells were not executed
    if "embed_model" not in globals() or "index" not in globals():

        embed_model = SentenceTransformer("all-MiniLM-L6-v2")
        corpus_text = (df["title"].fillna("") + " " + df["abstract"].fillna("")).tolist()
        embeddings = embed_model.encode(corpus_text, batch_size=32, show_progress_bar=False)
        faiss.normalize_L2(embeddings)
        index = faiss.IndexFlatIP(embeddings.shape[1])
        index.add(embeddings)

    top = search_papers(q, k=3)
    for r in top:
        print(f"  [{r['score']:.3f}] {r['title']}")


Query: vision transformers for image classification


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2641.57it/s]


  [0.505] An Unsupervised Algorithm For Learning Lie Group Transformations
  [0.500] Supervised Classification Performance of Multispectral Images
  [0.484] Binary Classification Based on Potentials

Query: reinforcement learning for robotics
  [0.581] Optimal Reinforcement Learning for Gaussian Systems
  [0.560] Closing the Learning-Planning Loop with Predictive State Representations
  [0.558] Feature Reinforcement Learning: Part I: Unstructured MDPs

Query: graph neural networks
  [0.452] Large Margin Boltzmann Machines and Large Margin Sigmoid Belief Networks
  [0.450] GraphLab: A New Framework for Parallel Machine Learning
  [0.446] Learning Undirected Graphical Models with Structure Penalty


## 9. LLM Agent

The search, summarization, and keyword tools are exposed to an LLM (Llama 3.1 8B via Groq)
through LangChain, so a user can ask questions in plain English and the agent decides which
tool(s) to call.

> **Note:** requires a `GROQ_API_KEY`. In Colab this is read from `google.colab.userdata`;
> outside Colab, set it as an environment variable instead.

In [31]:
import os

try:
    from google.colab import userdata  # type: ignore
    groq_api_key = userdata.get("GROQ_API_KEY")
except Exception:
    groq_api_key = os.environ.get("GROQ_API_KEY")

if not groq_api_key:
    try:
        from google.colab import userdata  # type: ignore
        groq_api_key = userdata.get("GROQ_API_KEY")
    except Exception:
        groq_api_key = os.environ.get("GROQ_API_KEY")

    if not groq_api_key:
        print(
            "GROQ_API_KEY not found. Set it in Colab secrets or as an environment variable."
        )


GROQ_API_KEY not found. Set it in Colab secrets or as an environment variable.


In [37]:
%pip install -q langchain-groq

import os
try:
    from langchain_groq import ChatGroq
except ImportError as exc:
    raise ImportError("langchain_groq is not installed.") from exc

from langchain_core.tools import tool

if groq_api_key is None:
    groq_api_key = os.environ.get("GROQ_API_KEY")

if not groq_api_key:
    print(
        "GROQ_API_KEY is not set. Set the environment variable or rerun the setup cell "
        "that populates `groq_api_key` before creating the ChatGroq client."
    )
    llm = None
else:
    llm = ChatGroq(model="llama-3.1-8b-instant", api_key=groq_api_key, temperature=0)

Note: you may need to restart the kernel to use updated packages.
GROQ_API_KEY is not set. Set the environment variable or rerun the setup cell that populates `groq_api_key` before creating the ChatGroq client.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [38]:
@tool
def search_and_summarize_tool(query: str, k: int = 5) -> str:
    """
    Search research papers by semantic similarity to the query, and return
    the top-k results along with a short generated summary of each.
    """
    try:
        results = search_and_summarize(query, k=k)
    except Exception as e:
        return f"Search failed: {e}"

    if not results:
        return "No matching papers were found."

    output = []
    for r in results:
        output.append(
            f"Title: {r['title']}\nSimilarity: {r['score']}\nSummary: {r['summary']}"
        )
    return "\n\n".join(output)


@tool
def extract_keywords_tool(text: str, top_n: int = 5) -> str:
    """
    Extract the most important keywords/keyphrases from a given piece of text.
    """
    try:
        keywords = extract_keywords(text, top_n=top_n)
    except Exception as e:
        return f"Keyword extraction failed: {e}"

    if not keywords:
        return "No keywords could be extracted."

    return "\n".join(f"{i+1}. {kw} (score: {score:.3f})" for i, (kw, score) in enumerate(keywords))


@tool
def compare_papers_tool(paper1: str, paper2: str) -> str:
    """
    Compare two research papers (identified by a title or topic description for each)
    based on their retrieved abstracts, across objective, methodology, contributions,
    advantages, limitations, and applications.
    """
    try:
        first = search_papers(paper1, k=1)[0]
        second = search_papers(paper2, k=1)[0]
    except Exception as e:
        return f"Could not retrieve one of the papers: {e}"

    comparison_prompt = f"""Compare the following two research papers.

Paper 1
Title: {first['title']}
Abstract: {first['abstract']}

Paper 2
Title: {second['title']}
Abstract: {second['abstract']}

Compare them based on:
1. Research Objective
2. Methodology
3. Key Contributions
4. Advantages
5. Limitations
6. Applications

Present the comparison as a clear table."""

    response = llm.invoke(comparison_prompt)
    return response.content


tools = [search_and_summarize_tool, extract_keywords_tool, compare_papers_tool]

In [49]:
from langchain_groq import ChatGroq

if llm is None:
    print("LLM client (llm) is not configured. Set GROQ_API_KEY and run the LangGroq setup cells before creating the agent.")
    agent = None
else:
    from langchain.agents import create_agent
    agent = create_agent(model=llm, tools=tools)

LLM client (llm) is not configured. Set GROQ_API_KEY and run the LangGroq setup cells before creating the agent.


## 10. Demo Queries

A few end-to-end examples showing the agent picking the right tool for each request.

In [52]:
if agent is None:
    print("Agent is not initialized. Please set GROQ_API_KEY environment variable or Colab secret and rerun the LLM setup cells.")
else:
    response = agent.invoke({
        "messages": [
            {"role": "user", "content": "Find the top 3 research papers on Vision Transformers and summarize them."}
        ]
    })
    print(response["messages"][-1].content)

Agent is not initialized. Please set GROQ_API_KEY environment variable or Colab secret and rerun the LLM setup cells.


In [55]:
if agent is None:
    print("Agent is not initialized. Set GROQ_API_KEY and rerun the LangChain/Groq setup cells.")
else:
    response = agent.invoke({
        "messages": [
            {"role": "user", "content": "Extract the top 5 keywords from: Deep Learning for Medical Image Reconstruction."}
        ]
    })
    print(response["messages"][-1].content)

if agent is None:
    print("Agent is not initialized. Set GROQ_API_KEY and rerun the LangChain/Groq setup cells.")
else:
    response = agent.invoke({
        "messages": [
            {"role": "user", "content": "Extract the top 5 keywords from: Deep Learning for Medical Image Reconstruction."}
        ]
    })
    print(response["messages"][-1].content)

Agent is not initialized. Set GROQ_API_KEY and rerun the LangChain/Groq setup cells.
Agent is not initialized. Set GROQ_API_KEY and rerun the LangChain/Groq setup cells.


In [58]:
if agent is None:
    print("Agent is not initialized. Set GROQ_API_KEY and rerun the LangChain/Groq setup cells.")
else:
    response = agent.invoke({
        "messages": [
            {"role": "user", "content": "Compare a paper about Vision Transformers with a paper about Convolutional Neural Networks."}
        ]
    })
    print(response["messages"][-1].content)

Agent is not initialized. Set GROQ_API_KEY and rerun the LangChain/Groq setup cells.


## 11. Summary

This notebook implements a full research-paper intelligence pipeline:

- **Retrieval:** semantic search over paper abstracts using Sentence-Transformer embeddings + FAISS
- **Summarization:** BART-based abstractive summarization with length scaled to input
- **Keyword extraction:** KeyBERT keyphrase extraction
- **Entity recognition:** general-purpose NER for models/datasets/institutions
- **Agent layer:** the above are exposed as LangChain tools so an LLM (Llama 3.1 via Groq) can
  answer natural-language questions end to end, choosing the right tool automatically

**Possible extensions:** swap in a domain-specific embedding model (e.g. SPECTER2), move to an
approximate FAISS index for larger corpora, add metadata filtering (year, category), and add a
lightweight FastAPI/Streamlit front end for interactive use.